# 12 · Proximal Policy Optimization
**DEEP RL** · ⏱ 50 min · 🧪 2 exercises · ⭐ 450 XP

*Reuse each batch for several careful updates — probability ratios, the clipped surrogate, value and entropy terms, and why PPO is the default.*

> **First time?** Open **00 · START HERE** in the same folder. Then run the cell below to connect this notebook to your Drive folder — your progress and XP are saved there.

In [ ]:
#@title ▶ **Run me first** — connect this notebook to your Reward Signal folder
#@markdown Mounts Google Drive and finds the `RewardSignal` folder (progress, XP and your files live there).
#@markdown Leave `ROOT` empty unless you put the folder somewhere unusual, e.g. `/content/drive/MyDrive/courses/RewardSignal`.
ROOT = ""  #@param {type:"string"}
LESSON = "12-ppo"
import os, sys

def _rs_locate(root=""):
    tries = [root, os.environ.get("RS_ROOT", "")]
    try:
        from google.colab import drive  # only exists in Colab
        if not os.path.isdir("/content/drive/MyDrive"):
            drive.mount("/content/drive")
        base = "/content/drive/MyDrive"
        tries += [f"{base}/RewardSignal", f"{base}/Colab Notebooks/RewardSignal", f"{base}/RewardSignal/RewardSignal"]
        tries += [os.path.join(base, d, "RewardSignal") for d in sorted(os.listdir(base)) if not d.startswith(".")]
        tries += ["/content/RewardSignal"]
    except ImportError:  # local Jupyter: the folder this notebook sits in
        tries += [os.getcwd(), os.path.dirname(os.getcwd())]
    for t in tries:
        if t and os.path.isfile(os.path.join(t, "runtime", "rlcolab.py")):
            return os.path.abspath(t)
    raise FileNotFoundError("Couldn't find the RewardSignal folder. Upload it to My Drive "
                            "(see its README.md) or type its full path into ROOT, then run this cell again.")

RS_ROOT = _rs_locate(ROOT.strip())
sys.dont_write_bytecode = True  # no __pycache__ folders in your Drive
if os.path.join(RS_ROOT, "runtime") not in sys.path:
    sys.path.insert(0, os.path.join(RS_ROOT, "runtime"))
import rlcolab
rlcolab.setup(RS_ROOT, lesson=LESSON)

Vanilla policy gradient and A2C have an awkward relationship with their data:

- **One step per batch.** The gradient Â · ∇log π is only valid *at the policy that collected
  the data*. After one update the policy has changed, so the batch is thrown away. Collecting
  experience is the expensive part — wasting it hurts.
- **Step-size roulette.** Too small and learning crawls. Too large and one update moves the
  policy somewhere terrible; the next batch is collected by that terrible policy, so there may
  be no way back. In supervised learning a bad step costs a little loss; in RL it can poison
  all future data.

PPO (Schulman et al., 2017) fixes both with one idea: **take several gradient steps on the
same batch, but never let the new policy drift far from the one that collected it.**

## 01 · Trust regions in one paragraph

TRPO, PPO's predecessor, maximised the expected advantage of the new policy *subject to* a
hard limit on how much it may change, measured by KL divergence:

```text
maximise  E[ (π_θ(a|s) / π_old(a|s)) Â ]     subject to     E[ KL(π_old ‖ π_θ) ] ≤ δ
```

That constraint defines a **trust region** — the neighbourhood where our advantage estimates,
collected under π_old, can still be trusted. TRPO enforces it with conjugate gradients and a
line search: effective, but complicated. PPO gets most of the benefit with a clipped objective
and plain SGD.

## 02 · The probability ratio

To reuse data from π_old while optimising π_θ, weight each sample by how much more (or less)
likely the new policy is to take that action — **importance sampling**:

```text
rₜ(θ) = π_θ(Aₜ | Sₜ) / π_old(Aₜ | Sₜ) = exp( log π_θ(Aₜ|Sₜ) − log π_old(Aₜ|Sₜ) )
```

The **surrogate objective** L(θ) = E[ rₜ(θ) Âₜ ] has exactly the policy gradient as its
gradient at θ = θ_old (where r = 1 and ∇r = ∇log π). Further away it is still a reasonable
estimate — as long as r stays close to 1. Maximising it without limits would happily push
r → ∞ for every positive-advantage action. That is the failure PPO prevents.

## 03 · The clipped surrogate

```text
L^CLIP(θ) = E[ min( rₜ Âₜ ,  clip(rₜ, 1 − ε, 1 + ε) Âₜ ) ]          ε ≈ 0.2
```

Read it case by case:

| advantage | ratio | what min() picks | gradient |
|---|---|---|---|
| Â > 0 (good action) | r ≤ 1 + ε | r·Â | push π(a) up |
| Â > 0 | r > 1 + ε | (1 + ε)·Â — constant | **0**: already up enough |
| Â < 0 (bad action) | r ≥ 1 − ε | r·Â | push π(a) down |
| Â < 0 | r < 1 − ε | (1 − ε)·Â — constant | **0**: already down enough |

The min makes the objective **pessimistic**: clipping only ever removes the incentive to move
*further* in the helpful direction. If an update accidentally moved the wrong way (Â > 0 but
r < 1), the unclipped term is the smaller one and its gradient pulls the policy back.

### Worked example

π_old(a|s) = 0.5, ε = 0.2, Â = +1.

- π_θ(a|s) = 0.55 → r = 1.1 → min(1.1, 1.1) = 1.1, gradient flows
- π_θ(a|s) = 0.75 → r = 1.5 → min(1.5, 1.2) = **1.2**, flat: no more push
- π_θ(a|s) = 0.25 → r = 0.5 → min(0.5, 0.8) = **0.5**, gradient flows (undo the mistake)

In [ ]:
#@title 🧠 **Knowledge check 1** · +25 XP — run, then pick an answer
rlcolab.quiz("12-ppo#0", {"q": "Â = −2, ε = 0.2, and after a few updates r = 0.7 for this sample. What is its contribution to L^CLIP and its gradient?", "options": ["−1.4, gradient pushes π(a) further down", "−1.6, zero gradient", "−1.4, zero gradient", "−1.6, gradient pushes π(a) up"], "answer": 1, "explain": "min(0.7·(−2), 0.8·(−2)) = min(−1.4, −1.6) = −1.6 — the clipped, constant term. The bad action is already 30% less likely, more than the 20% PPO allows credit for, so it stops pushing."})

---
### 🧪 Exercise 1 · The clipped surrogate objective · ⭐ **150 XP**

Implement the heart of PPO.

`ppo_clip_loss(logp_new, logp_old, adv, clip_eps)` returns `(loss, grad)`:
- ratio **rₜ = exp(logp_new − logp_old)**
- **loss = −mean(min(rₜÂₜ, clip(rₜ, 1−ε, 1+ε)Âₜ))** (negated because we minimise)
- **grad** = ∂loss/∂logp_new, same shape as `logp_new`. Since d rₜ / d logp_new = rₜ, it is −rₜÂₜ/N — except where the *clipped* term is the minimum, where the objective is flat and the gradient is 0. That happens when Â > 0 and r > 1+ε, or Â < 0 and r < 1−ε.

`ppo_diagnostics(logp_new, logp_old, clip_eps)` returns `(approx_kl, clip_fraction)` with approx_kl = mean((r − 1) − log r).

The check walks through all six sign/clip cases. The demo plots the objective and its gradient against the ratio.

> ✏️ **Edit** the next cell and run it (`Shift+Enter`) — that also runs its demo. Then run **✅ Check**. Stuck? The hints are right below the check.

In [ ]:
# ✏️ EXERCISE 12-ppo/01-clipped-objective
import numpy as np
import rlhud


def ppo_clip_loss(logp_new, logp_old, adv, clip_eps=0.2):
    """PPO's clipped surrogate, written as a LOSS to minimise, and its gradient.

    ratio rₜ = π_new(aₜ|sₜ) / π_old(aₜ|sₜ) = exp(logp_new − logp_old)
    loss     = −mean_t[ min( rₜ Âₜ,  clip(rₜ, 1−ε, 1+ε) Âₜ ) ]
    returns (loss, dloss/dlogp_new) — the gradient has the shape of logp_new.

    Where the clipped term is the smaller one (Â > 0 and r > 1+ε, or Â < 0 and r < 1−ε)
    the objective is flat in θ, so the gradient is 0. Elsewhere d(r Â)/dlogp_new = r Â.
    """
    # TODO
    # 1) ratio = exp(logp_new − logp_old)
    # 2) loss = −mean(min(ratio·adv, clip(ratio, 1−ε, 1+ε)·adv))
    # 3) grad = −ratio·adv / N, except 0 where the clipped term is the active (flat) one
    raise NotImplementedError


def ppo_diagnostics(logp_new, logp_old, clip_eps=0.2):
    """Two numbers every PPO implementation logs. Returns (approx_kl, clip_fraction).

    approx_kl     = mean( (r − 1) − log r )     — a low-variance estimate of KL(π_old ‖ π_new), ≥ 0
    clip_fraction = fraction of samples with |r − 1| > ε
    """
    # TODO
    raise NotImplementedError


if __name__ == "__main__":
    # The objective for ONE sample as a function of the ratio r, for a positive and a negative
    # advantage. Plain policy gradient keeps rising forever; PPO's objective goes flat once the
    # policy has moved "far enough" in the helpful direction.
    rs = np.linspace(0.025, 2.0, 80)
    for A in (1.0, -1.0):
        obj = [-ppo_clip_loss(np.log([r]), [0.0], [A])[0] for r in rs]
        rlhud.plot(obj, x=rs, label=f"PPO, Â = {A:+.0f}", title="Surrogate objective vs ratio r", xlabel="r")
        rlhud.plot(rs * A, x=rs, label=f"unclipped r·Â, Â = {A:+.0f}", title="Surrogate objective vs ratio r", xlabel="r")
        grads = [-ppo_clip_loss(np.log([r]), [0.0], [A])[1][0] for r in rs]
        rlhud.plot(grads, x=rs, label=f"Â = {A:+.0f}", title="d objective / d log π_new", xlabel="r")
    kl, cf = ppo_diagnostics(np.log([0.5, 0.7, 0.9]), np.log([0.4, 0.7, 0.6]))
    rlhud.stat("approx KL (example)", kl)
    rlhud.stat("clip fraction (example)", cf)

In [ ]:
#@title ✅ **Check** · The clipped surrogate objective (+150 XP) — tests the code you last ran above
rlcolab.check("12-ppo/01-clipped-objective")

<details><summary>💡 <b>Hint 1</b> of 3 — try it before opening the next one</summary>

`ratio = np.exp(logp_new - logp_old)`; `loss = -np.mean(np.minimum(ratio * adv, np.clip(ratio, 1 - eps, 1 + eps) * adv))`.


<details><summary>💡 <b>Hint 2</b> of 3 — try it before opening the next one</summary>

`flat = ((adv > 0) & (ratio > 1 + eps)) | ((adv < 0) & (ratio < 1 - eps))` marks samples whose gradient is zero.


<details><summary>💡 <b>Hint 3</b> of 3 — nearly gives it away</summary>

`grad = np.where(flat, 0.0, -ratio * adv) / len(adv)`. Diagnostics: `log_r = logp_new - logp_old; r = np.exp(log_r)`.

</details>

</details>

</details>

In [ ]:
#@title 🔒 Reference solution — double-click to reveal · running it replaces your version
rlcolab.solution_gate("12-ppo/01-clipped-objective")  # obeys the Solutions setting in 00 · START HERE
# ✏️ EXERCISE 12-ppo/01-clipped-objective
# Reference solution
import numpy as np
import rlhud


def ppo_clip_loss(logp_new, logp_old, adv, clip_eps=0.2):
    """PPO's clipped surrogate, written as a LOSS to minimise, and its gradient.

    ratio rₜ = π_new(aₜ|sₜ) / π_old(aₜ|sₜ) = exp(logp_new − logp_old)
    loss     = −mean_t[ min( rₜ Âₜ,  clip(rₜ, 1−ε, 1+ε) Âₜ ) ]
    returns (loss, dloss/dlogp_new) — the gradient has the shape of logp_new.

    Where the clipped term is the smaller one (Â > 0 and r > 1+ε, or Â < 0 and r < 1−ε)
    the objective is flat in θ, so the gradient is 0. Elsewhere d(r Â)/dlogp_new = r Â.
    """
    logp_new, logp_old, adv = (np.asarray(v, dtype=float) for v in (logp_new, logp_old, adv))
    ratio = np.exp(logp_new - logp_old)
    unclipped = ratio * adv
    clipped = np.clip(ratio, 1 - clip_eps, 1 + clip_eps) * adv
    loss = -np.mean(np.minimum(unclipped, clipped))
    flat = ((adv > 0) & (ratio > 1 + clip_eps)) | ((adv < 0) & (ratio < 1 - clip_eps))
    grad = np.where(flat, 0.0, -ratio * adv) / len(adv)
    return float(loss), grad


def ppo_diagnostics(logp_new, logp_old, clip_eps=0.2):
    """Two numbers every PPO implementation logs. Returns (approx_kl, clip_fraction).

    approx_kl     = mean( (r − 1) − log r )     — a low-variance estimate of KL(π_old ‖ π_new), ≥ 0
    clip_fraction = fraction of samples with |r − 1| > ε
    """
    log_r = np.asarray(logp_new, dtype=float) - np.asarray(logp_old, dtype=float)
    r = np.exp(log_r)
    return float(np.mean((r - 1) - log_r)), float(np.mean(np.abs(r - 1) > clip_eps))


if __name__ == "__main__":
    # The objective for ONE sample as a function of the ratio r, for a positive and a negative
    # advantage. Plain policy gradient keeps rising forever; PPO's objective goes flat once the
    # policy has moved "far enough" in the helpful direction.
    rs = np.linspace(0.025, 2.0, 80)
    for A in (1.0, -1.0):
        obj = [-ppo_clip_loss(np.log([r]), [0.0], [A])[0] for r in rs]
        rlhud.plot(obj, x=rs, label=f"PPO, Â = {A:+.0f}", title="Surrogate objective vs ratio r", xlabel="r")
        rlhud.plot(rs * A, x=rs, label=f"unclipped r·Â, Â = {A:+.0f}", title="Surrogate objective vs ratio r", xlabel="r")
        grads = [-ppo_clip_loss(np.log([r]), [0.0], [A])[1][0] for r in rs]
        rlhud.plot(grads, x=rs, label=f"Â = {A:+.0f}", title="d objective / d log π_new", xlabel="r")
    kl, cf = ppo_diagnostics(np.log([0.5, 0.7, 0.9]), np.log([0.4, 0.7, 0.6]))
    rlhud.stat("approx KL (example)", kl)
    rlhud.stat("clip fraction (example)", cf)

## 04 · The full PPO loss

PPO is an actor-critic. With a shared optimiser (or separate ones, as in the exercise) the
total loss to minimise is

```text
L = −L^CLIP(θ)  +  c_v · ½ E[ (v̂(Sₜ) − targetₜ)² ]  −  c_e · E[ H(π_θ(·|Sₜ)) ]
```

- **Value loss** — the critic regresses onto the GAE λ-returns (Âₜ + v̂_old(Sₜ)).
- **Entropy bonus** — H(π) = −Σ π log π rewards keeping the policy stochastic, delaying
  premature collapse. c_e ≈ 0.01 for discrete actions; often 0 for continuous ones.

## 05 · The algorithm

```python
for iteration in range(...):
    # 1. collect N steps (e.g. 512 here; 2048 × 8 envs in big runs) with π_old
    #    store s, a, log π_old(a|s), r, done
    # 2. GAE: advantages Â and value targets  (λ = 0.95, γ = 0.99)
    for epoch in range(K):                    # K ≈ 4–10
        for minibatch in shuffle(batch, size=64):
            normalise Â in the minibatch
            ratio = exp(log π_θ(a|s) − log π_old(a|s))
            actor:  minimise −min(r·Â, clip(r)·Â) − c_e · H
            critic: minimise ½ (v̂(s) − target)²
```

To backprop through a softmax actor you only need one fact from lesson 10:
∂ log π(a|s) / ∂ logits = 𝟙ₐ − π. The clipped loss gives ∂L/∂ log π, and the chain rule does
the rest.

In [ ]:
#@title 🧠 **Knowledge check 2** · +25 XP — run, then pick an answer
rlcolab.quiz("12-ppo#1", {"q": "Why does PPO store log π_old(a|s) at collection time instead of recomputing it later?", "options": ["To save computation only", "Because the policy changes during the K epochs — the ratio must compare against the policy that actually collected the data", "Because log-probabilities can't be recomputed for discrete actions", "It's needed for GAE"], "answer": 1, "explain": "After the first minibatch step π_θ ≠ π_old. The ratio is only meaningful relative to the frozen behaviour policy, so its log-probabilities are recorded when acting."})

---
### 🧪 Exercise 2 · PPO on CartPole · ⭐ **300 XP**

Build a compact **Proximal Policy Optimization** agent: an MLP actor and an MLP critic, rollouts of 512 steps, 4 epochs of minibatch updates per rollout.

Implement:
- `ppo_clip_loss(...)` — paste your solution from the previous exercise.
- `ppo_update(actor, critic, actor_opt, critic_opt, batch, rng, ...)` — for each epoch, shuffle the batch and, for each minibatch:
  1. normalise the advantages,
  2. **actor**: logp = log π(a|s) from `softmax(actor.forward(obs))`; get `dlogp` from `ppo_clip_loss`; chain-rule into the logits with **dlogits = dlogp · (onehot − π)**; add `ent_coef * entropy_loss_grad(P)`; `actor_opt.step(actor.backward(dlogits))`,
  3. **critic**: ½-MSE regression of `critic.forward(obs)` onto `returns · value_scale`,
  4. record the approximate KL; return the mean KL of the last epoch.

Networks, Adam, GAE, the entropy gradient and the rollout loop are provided. The check runs one update on a synthetic batch, then trains for 80 000 steps (≈ 6 s) and requires a mean return ≥ 250.

> ✏️ **Edit** the next cell and run it (`Shift+Enter`) — that also runs its demo. Then run **✅ Check**. Stuck? The hints are right below the check.

In [ ]:
# ✏️ EXERCISE 12-ppo/02-ppo-cartpole
import numpy as np
import rlhud
from rlenvs import CartPole


# ---------------------------------------------------------------- provided: networks, optimiser, helpers
class MLP:
    """Linear → tanh → … → Linear with manual backprop."""

    def __init__(self, sizes, seed=0, out_scale=1.0):
        rng = np.random.default_rng(seed)
        self.params = []
        for i, (n_in, n_out) in enumerate(zip(sizes[:-1], sizes[1:])):
            scale = np.sqrt(1.0 / n_in) * (out_scale if i == len(sizes) - 2 else 1.0)
            self.params += [rng.normal(0.0, scale, size=(n_in, n_out)), np.zeros(n_out)]
        self.n_layers = len(sizes) - 1

    def forward(self, x):
        h, self._inputs = np.asarray(x, dtype=float), []
        for i in range(self.n_layers):
            self._inputs.append(h)
            h = h @ self.params[2 * i] + self.params[2 * i + 1]
            if i < self.n_layers - 1:
                h = np.tanh(h)
        return h

    def backward(self, dout):
        grads, d = [None] * len(self.params), np.asarray(dout, dtype=float)
        for i in reversed(range(self.n_layers)):
            grads[2 * i], grads[2 * i + 1] = self._inputs[i].T @ d, d.sum(axis=0)
            if i > 0:
                d = (d @ self.params[2 * i].T) * (1.0 - self._inputs[i] ** 2)
        return grads


class Adam:
    def __init__(self, params, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8):
        self.params, self.lr, self.b1, self.b2, self.eps = params, lr, beta1, beta2, eps
        self.m = [np.zeros_like(p) for p in params]
        self.v = [np.zeros_like(p) for p in params]
        self.t = 0

    def step(self, grads):
        self.t += 1
        for p, g, m, v in zip(self.params, grads, self.m, self.v):
            m *= self.b1; m += (1 - self.b1) * g
            v *= self.b2; v += (1 - self.b2) * g * g
            p -= self.lr * (m / (1 - self.b1 ** self.t)) / (np.sqrt(v / (1 - self.b2 ** self.t)) + self.eps)


def softmax(z):
    e = np.exp(z - z.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)


def entropy_loss_grad(P):
    """Gradient w.r.t. the logits of  −mean(H(π)),  H = −Σ p log p.  Shape like P.
    Adding  ent_coef · this  to the actor's logit-gradient implements the entropy bonus."""
    logP = np.log(P + 1e-12)
    H = -(P * logP).sum(axis=1, keepdims=True)
    return P * (logP + H) / len(P)


def compute_gae(rewards, values, dones, last_value, gamma=0.99, lam=0.95):
    rewards, values, dones = (np.asarray(x, dtype=float) for x in (rewards, values, dones))
    deltas = rewards + gamma * (1.0 - dones) * np.append(values[1:], last_value) - values
    adv, running = np.zeros(len(deltas)), 0.0
    for t in reversed(range(len(deltas))):
        running = deltas[t] + gamma * lam * (1.0 - dones[t]) * running
        adv[t] = running
    return adv, adv + values


# ---------------------------------------------------------------- your code
def ppo_clip_loss(logp_new, logp_old, adv, clip_eps=0.2):
    """(loss, dloss/dlogp_new) — paste your solution from the previous exercise."""
    # TODO
    raise NotImplementedError


def ppo_update(actor, critic, actor_opt, critic_opt, batch, rng, epochs=4, minibatch_size=64,
               clip_eps=0.2, ent_coef=0.01, value_scale=0.01):
    """Several epochs of minibatch updates on one rollout. Returns the mean approx-KL of the
    last epoch (a health statistic).

    batch: dict of arrays with N rows — "obs" (N, 4), "actions" (N,), "logp_old" (N,),
           "advantages" (N,), "returns" (N,)
    """
    N = len(batch["actions"])
    kls = []
    for epoch in range(epochs):
        kls = []
        order = rng.permutation(N)
        for start in range(0, N, minibatch_size):
            mb = order[start:start + minibatch_size]
            obs, act = batch["obs"][mb], batch["actions"][mb]
            # TODO 1: normalise the minibatch advantages to mean 0, std 1
            # TODO 2: actor
            #   P = softmax(actor.forward(obs)); logp = log P[row, act]
            #   _, dlogp = ppo_clip_loss(logp, batch["logp_old"][mb], adv, clip_eps)
            #   chain rule through log-softmax: dlogits = dlogp[:, None] * (onehot(act) − P)
            #   entropy bonus: dlogits += ent_coef * entropy_loss_grad(P)
            #   actor_opt.step(actor.backward(dlogits))
            # TODO 3: critic — regress critic.forward(obs) onto returns·value_scale (½ MSE)
            # TODO 4: kls.append(approx KL of this minibatch) — see ppo_diagnostics
            raise NotImplementedError
    return float(np.mean(kls))


# ---------------------------------------------------------------- provided: rollouts + training loop
def train(total_steps=80_000, seed=0, n_steps=512, gamma=0.99, lam=0.95, lr=1e-3, hidden=32,
          epochs=4, minibatch_size=64, clip_eps=0.2, ent_coef=0.01, value_scale=0.01, log=None):
    """PPO on CartPole. Returns (actor, critic, episode_returns)."""
    rng = np.random.default_rng(seed)
    env = CartPole(seed=seed)
    actor = MLP([4, hidden, env.n_actions], seed=seed, out_scale=0.01)   # near-uniform initial policy
    critic = MLP([4, hidden, 1], seed=seed + 1)
    actor_opt, critic_opt = Adam(actor.params, lr), Adam(critic.params, lr)
    V = lambda o: critic.forward(np.atleast_2d(o))[:, 0] / value_scale

    obs, _ = env.reset(seed=seed)
    ep_return, returns, t = 0.0, [], 0
    while t < total_steps:
        # 1) collect n_steps with the current ("old") policy, remembering log π_old(a|s)
        O, A, LP, R, D = [], [], [], [], []
        for _ in range(n_steps):
            p = softmax(actor.forward(obs[None, :]))[0]
            a = min(int(np.searchsorted(np.cumsum(p), rng.random(), side="right")), len(p) - 1)
            obs2, r, terminated, truncated, _ = env.step(a)
            if truncated and not terminated:
                r += gamma * V(obs2)[0]                  # time-limit bootstrap
            O.append(obs); A.append(a); LP.append(np.log(p[a])); R.append(r)
            D.append(float(terminated or truncated))
            ep_return += 1.0; t += 1
            obs = obs2
            if terminated or truncated:
                returns.append(ep_return)
                if log:
                    log(len(returns), ep_return, None)
                ep_return = 0.0
                obs, _ = env.reset()
        O = np.array(O)
        # 2) advantages and value targets
        adv, rets = compute_gae(R, V(O), D, V(obs)[0], gamma, lam)
        batch = {"obs": O, "actions": np.array(A), "logp_old": np.array(LP), "advantages": adv, "returns": rets}
        # 3) several epochs of clipped updates on this one batch
        kl = ppo_update(actor, critic, actor_opt, critic_opt, batch, rng, epochs, minibatch_size,
                        clip_eps, ent_coef, value_scale)
        if log:
            log(len(returns), None, kl)
    return actor, critic, returns


def evaluate(actor, episodes=10, seed=1000):
    """Mean return of the stochastic policy."""
    env, rng, total = CartPole(), np.random.default_rng(123), 0.0
    for i in range(episodes):
        obs, _ = env.reset(seed=seed + i)
        done = False
        while not done:
            p = softmax(actor.forward(obs[None, :]))[0]
            a = min(int(np.searchsorted(np.cumsum(p), rng.random(), side="right")), len(p) - 1)
            obs, r, term, trunc, _ = env.step(a)
            total += r
            done = term or trunc
    return total / episodes


if __name__ == "__main__":
    live = rlhud.chart("PPO on CartPole — episode return", smooth=10, xlabel="episode")
    kl_chart = rlhud.chart("Approx KL per update (keep it small: ~0.001–0.02)", xlabel="update")

    def log(ep, G, kl):
        if G is not None:
            live.add("PPO", G)
        if kl is not None:
            kl_chart.add("KL", kl)

    actor, critic, returns = train(total_steps=80_000, seed=0, log=log)
    live.flush(); kl_chart.flush()
    score = evaluate(actor)
    rlhud.stat("Evaluation (10 episodes)", score, good=score >= 200)
    rlhud.stat("Episodes", len(returns))

In [ ]:
#@title ✅ **Check** · PPO on CartPole (+300 XP) — tests the code you last ran above
rlcolab.check("12-ppo/02-ppo-cartpole")

<details><summary>💡 <b>Hint 1</b> of 3 — try it before opening the next one</summary>

Normalise: `adv = batch['advantages'][mb]; adv = (adv - adv.mean()) / (adv.std() + 1e-8)`. Rows for fancy indexing: `r = np.arange(len(mb))`, then `logp = np.log(P[r, act] + 1e-12)`.


<details><summary>💡 <b>Hint 2</b> of 3 — try it before opening the next one</summary>

`_, dlogp = ppo_clip_loss(logp, batch['logp_old'][mb], adv, clip_eps)`; `onehot = np.eye(P.shape[1])[act]`; `dlogits = dlogp[:, None] * (onehot - P) + ent_coef * entropy_loss_grad(P)`.


<details><summary>💡 <b>Hint 3</b> of 3 — nearly gives it away</summary>

Critic: `v = critic.forward(obs); target = batch['returns'][mb][:, None] * value_scale; critic_opt.step(critic.backward((v - target) / len(mb)))`. KL: `lr_ = logp - batch['logp_old'][mb]; kls.append(np.mean(np.exp(lr_) - 1 - lr_))`.

</details>

</details>

</details>

In [ ]:
#@title 🔒 Reference solution — double-click to reveal · running it replaces your version
rlcolab.solution_gate("12-ppo/02-ppo-cartpole")  # obeys the Solutions setting in 00 · START HERE
# ✏️ EXERCISE 12-ppo/02-ppo-cartpole
# Reference solution
import numpy as np
import rlhud
from rlenvs import CartPole


# ---------------------------------------------------------------- provided: networks, optimiser, helpers
class MLP:
    """Linear → tanh → … → Linear with manual backprop."""

    def __init__(self, sizes, seed=0, out_scale=1.0):
        rng = np.random.default_rng(seed)
        self.params = []
        for i, (n_in, n_out) in enumerate(zip(sizes[:-1], sizes[1:])):
            scale = np.sqrt(1.0 / n_in) * (out_scale if i == len(sizes) - 2 else 1.0)
            self.params += [rng.normal(0.0, scale, size=(n_in, n_out)), np.zeros(n_out)]
        self.n_layers = len(sizes) - 1

    def forward(self, x):
        h, self._inputs = np.asarray(x, dtype=float), []
        for i in range(self.n_layers):
            self._inputs.append(h)
            h = h @ self.params[2 * i] + self.params[2 * i + 1]
            if i < self.n_layers - 1:
                h = np.tanh(h)
        return h

    def backward(self, dout):
        grads, d = [None] * len(self.params), np.asarray(dout, dtype=float)
        for i in reversed(range(self.n_layers)):
            grads[2 * i], grads[2 * i + 1] = self._inputs[i].T @ d, d.sum(axis=0)
            if i > 0:
                d = (d @ self.params[2 * i].T) * (1.0 - self._inputs[i] ** 2)
        return grads


class Adam:
    def __init__(self, params, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8):
        self.params, self.lr, self.b1, self.b2, self.eps = params, lr, beta1, beta2, eps
        self.m = [np.zeros_like(p) for p in params]
        self.v = [np.zeros_like(p) for p in params]
        self.t = 0

    def step(self, grads):
        self.t += 1
        for p, g, m, v in zip(self.params, grads, self.m, self.v):
            m *= self.b1; m += (1 - self.b1) * g
            v *= self.b2; v += (1 - self.b2) * g * g
            p -= self.lr * (m / (1 - self.b1 ** self.t)) / (np.sqrt(v / (1 - self.b2 ** self.t)) + self.eps)


def softmax(z):
    e = np.exp(z - z.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)


def entropy_loss_grad(P):
    """Gradient w.r.t. the logits of  −mean(H(π)),  H = −Σ p log p.  Shape like P.
    Adding  ent_coef · this  to the actor's logit-gradient implements the entropy bonus."""
    logP = np.log(P + 1e-12)
    H = -(P * logP).sum(axis=1, keepdims=True)
    return P * (logP + H) / len(P)


def compute_gae(rewards, values, dones, last_value, gamma=0.99, lam=0.95):
    rewards, values, dones = (np.asarray(x, dtype=float) for x in (rewards, values, dones))
    deltas = rewards + gamma * (1.0 - dones) * np.append(values[1:], last_value) - values
    adv, running = np.zeros(len(deltas)), 0.0
    for t in reversed(range(len(deltas))):
        running = deltas[t] + gamma * lam * (1.0 - dones[t]) * running
        adv[t] = running
    return adv, adv + values


# ---------------------------------------------------------------- your code
def ppo_clip_loss(logp_new, logp_old, adv, clip_eps=0.2):
    """(loss, dloss/dlogp_new) — paste your solution from the previous exercise."""
    ratio = np.exp(logp_new - logp_old)
    loss = -np.mean(np.minimum(ratio * adv, np.clip(ratio, 1 - clip_eps, 1 + clip_eps) * adv))
    flat = ((adv > 0) & (ratio > 1 + clip_eps)) | ((adv < 0) & (ratio < 1 - clip_eps))
    return float(loss), np.where(flat, 0.0, -ratio * adv) / len(adv)


def ppo_update(actor, critic, actor_opt, critic_opt, batch, rng, epochs=4, minibatch_size=64,
               clip_eps=0.2, ent_coef=0.01, value_scale=0.01):
    """Several epochs of minibatch updates on one rollout. Returns the mean approx-KL of the
    last epoch (a health statistic).

    batch: dict of arrays with N rows — "obs" (N, 4), "actions" (N,), "logp_old" (N,),
           "advantages" (N,), "returns" (N,)
    """
    N = len(batch["actions"])
    rows = np.arange(minibatch_size)
    for epoch in range(epochs):
        kls = []
        order = rng.permutation(N)
        for start in range(0, N, minibatch_size):
            mb = order[start:start + minibatch_size]
            obs, act = batch["obs"][mb], batch["actions"][mb]
            adv = batch["advantages"][mb]
            adv = (adv - adv.mean()) / (adv.std() + 1e-8)          # per-minibatch normalisation

            # --- actor: clipped surrogate + entropy bonus, back-propagated through the softmax
            P = softmax(actor.forward(obs))
            logp = np.log(P[rows[:len(mb)], act] + 1e-12)
            _, dlogp = ppo_clip_loss(logp, batch["logp_old"][mb], adv, clip_eps)
            onehot = np.eye(P.shape[1])[act]
            dlogits = dlogp[:, None] * (onehot - P)                 # ∂ log π(a) / ∂ logits = onehot − π
            dlogits += ent_coef * entropy_loss_grad(P)
            actor_opt.step(actor.backward(dlogits))

            # --- critic: regression on the (scaled) λ-returns
            v = critic.forward(obs)
            target = batch["returns"][mb][:, None] * value_scale
            critic_opt.step(critic.backward((v - target) / len(mb)))
            kls.append(np.mean((np.exp(logp - batch["logp_old"][mb]) - 1) - (logp - batch["logp_old"][mb])))
    return float(np.mean(kls))


# ---------------------------------------------------------------- provided: rollouts + training loop
def train(total_steps=80_000, seed=0, n_steps=512, gamma=0.99, lam=0.95, lr=1e-3, hidden=32,
          epochs=4, minibatch_size=64, clip_eps=0.2, ent_coef=0.01, value_scale=0.01, log=None):
    """PPO on CartPole. Returns (actor, critic, episode_returns)."""
    rng = np.random.default_rng(seed)
    env = CartPole(seed=seed)
    actor = MLP([4, hidden, env.n_actions], seed=seed, out_scale=0.01)   # near-uniform initial policy
    critic = MLP([4, hidden, 1], seed=seed + 1)
    actor_opt, critic_opt = Adam(actor.params, lr), Adam(critic.params, lr)
    V = lambda o: critic.forward(np.atleast_2d(o))[:, 0] / value_scale

    obs, _ = env.reset(seed=seed)
    ep_return, returns, t = 0.0, [], 0
    while t < total_steps:
        # 1) collect n_steps with the current ("old") policy, remembering log π_old(a|s)
        O, A, LP, R, D = [], [], [], [], []
        for _ in range(n_steps):
            p = softmax(actor.forward(obs[None, :]))[0]
            a = min(int(np.searchsorted(np.cumsum(p), rng.random(), side="right")), len(p) - 1)
            obs2, r, terminated, truncated, _ = env.step(a)
            if truncated and not terminated:
                r += gamma * V(obs2)[0]                  # time-limit bootstrap
            O.append(obs); A.append(a); LP.append(np.log(p[a])); R.append(r)
            D.append(float(terminated or truncated))
            ep_return += 1.0; t += 1
            obs = obs2
            if terminated or truncated:
                returns.append(ep_return)
                if log:
                    log(len(returns), ep_return, None)
                ep_return = 0.0
                obs, _ = env.reset()
        O = np.array(O)
        # 2) advantages and value targets
        adv, rets = compute_gae(R, V(O), D, V(obs)[0], gamma, lam)
        batch = {"obs": O, "actions": np.array(A), "logp_old": np.array(LP), "advantages": adv, "returns": rets}
        # 3) several epochs of clipped updates on this one batch
        kl = ppo_update(actor, critic, actor_opt, critic_opt, batch, rng, epochs, minibatch_size,
                        clip_eps, ent_coef, value_scale)
        if log:
            log(len(returns), None, kl)
    return actor, critic, returns


def evaluate(actor, episodes=10, seed=1000):
    """Mean return of the stochastic policy."""
    env, rng, total = CartPole(), np.random.default_rng(123), 0.0
    for i in range(episodes):
        obs, _ = env.reset(seed=seed + i)
        done = False
        while not done:
            p = softmax(actor.forward(obs[None, :]))[0]
            a = min(int(np.searchsorted(np.cumsum(p), rng.random(), side="right")), len(p) - 1)
            obs, r, term, trunc, _ = env.step(a)
            total += r
            done = term or trunc
    return total / episodes


if __name__ == "__main__":
    live = rlhud.chart("PPO on CartPole — episode return", smooth=10, xlabel="episode")
    kl_chart = rlhud.chart("Approx KL per update (keep it small: ~0.001–0.02)", xlabel="update")

    def log(ep, G, kl):
        if G is not None:
            live.add("PPO", G)
        if kl is not None:
            kl_chart.add("KL", kl)

    actor, critic, returns = train(total_steps=80_000, seed=0, log=log)
    live.flush(); kl_chart.flush()
    score = evaluate(actor)
    rlhud.stat("Evaluation (10 episodes)", score, good=score >= 200)
    rlhud.stat("Episodes", len(returns))

> **🔑 KEY IDEA** · **Many small, safe steps from one batch**
>
> PPO squeezes several epochs of SGD out of each rollout, and the clipped ratio removes any
> incentive to move the policy more than ~ε from the one that collected the data. Data
> efficiency of off-policy reuse, stability of a trust region, simplicity of plain SGD.

## 06 · Why PPO is the default workhorse

- **Robust.** The same hyperparameters (ε = 0.2, λ = 0.95, γ = 0.99, lr ≈ 3e-4, 4–10 epochs)
  work across Atari, robotics simulators and board games with little tuning.
- **Simple.** No replay buffer, no target networks, no second-order optimisation — about 300
  lines in a single file (see CleanRL's `ppo.py`).
- **General.** Discrete or continuous actions (swap the softmax for a Gaussian), recurrent
  policies, many parallel environments.
- **Scales.** OpenAI Five (Dota 2) and the RLHF stage that fine-tunes large language models
  both used PPO.

Its weakness is **sample efficiency**: it is on-policy, so every batch is discarded after K
epochs. When environment steps are expensive (real robots), off-policy methods win.

## 07 · Debugging PPO

Log these four numbers every update — they tell you more than the return curve:

- **approx KL** between π_old and π_θ: typically 0.001–0.02. Much larger → lower the learning
  rate or the number of epochs (many implementations stop the epochs early if KL > ~0.015).
  Near zero → the actor is not learning: check the gradient chain.
- **clip fraction**: share of samples with |r − 1| > ε. Around 0.05–0.2 is healthy.
- **entropy**: should decline slowly, not crash to 0 in a few iterations.
- **explained variance** of the critic: should rise toward 1.

And the classic bugs: forgetting to normalise advantages, normalising *returns* used as value
targets inconsistently, a stale `logp_old` (recomputed after updating), mixing up the sign of
the loss, and bootstrapping through `terminated`.

In [ ]:
#@title 🧠 **Knowledge check 3** · +25 XP — run, then pick an answer
rlcolab.quiz("12-ppo#2", {"q": "Your PPO logs show approx KL ≈ 0.3 per update and the return oscillates wildly. What is the best first fix?", "options": ["Increase ε to 0.5", "Lower the learning rate and/or the number of epochs", "Remove the entropy bonus", "Use a larger minibatch only"], "answer": 1, "explain": "KL that large means each update moves the policy far outside the trust region — the clip can't fully stop many epochs of large steps. Smaller lr / fewer epochs (or KL early stopping) restores 'proximal'."})

## 08 · If you have PyTorch

```python
dist = torch.distributions.Categorical(logits=actor(obs_mb))
logp = dist.log_prob(act_mb)
ratio = torch.exp(logp - logp_old_mb)
adv = (adv_mb - adv_mb.mean()) / (adv_mb.std() + 1e-8)
pg_loss = -torch.min(ratio * adv, torch.clamp(ratio, 1 - 0.2, 1 + 0.2) * adv).mean()
v_loss = 0.5 * (critic(obs_mb).squeeze(-1) - ret_mb).pow(2).mean()
loss = pg_loss + 0.5 * v_loss - 0.01 * dist.entropy().mean()
opt.zero_grad(); loss.backward()
nn.utils.clip_grad_norm_(params, 0.5); opt.step()
```

## 09 · Where to go next

You have now built, from scratch, the core of modern deep RL. Some directions from here:

- **Continuous control.** Replace the softmax with a Gaussian policy (mean from the network,
  learned log-std) and PPO works on MuJoCo-style robots. The strongest off-policy alternatives
  are **SAC** (Soft Actor-Critic — maximum-entropy RL with twin Q-networks) and **TD3** (DDPG
  plus clipped double-Q and delayed policy updates).
- **Gymnasium** (the maintained successor of OpenAI Gym) — the standard environment API you
  have been using all along: `gymnasium.make("CartPole-v1")` runs your code unchanged.
  Add `gymnasium[mujoco]` or `ale-py` for robotics and Atari.
- **Stable-Baselines3** — reliable PyTorch implementations of PPO, A2C, DQN, SAC, TD3 with a
  three-line API. Use it as a baseline before trusting your own implementation.
- **CleanRL** — single-file, readable implementations of every major algorithm; ideal for
  reading after this course.
- **Beyond:** distributional RL (C51, QR-DQN), Rainbow, model-based RL (MuZero, Dreamer),
  offline RL (CQL, IQL), multi-agent RL, and RL from human feedback (PPO/DPO for language models).
- **Books & courses:** Sutton & Barto, *Reinforcement Learning: An Introduction* (2nd ed.,
  free online); OpenAI's *Spinning Up in Deep RL*; David Silver's UCL lectures.

---
## 🏁 Debrief
Run the cell below for your mission status, then open the next notebook.

In [ ]:
#@title 🏁 **Mission status** — run to refresh
rlcolab.lesson_footer("12-ppo")